# IV Rank Predictor — Train on Kaggle (Free T4 GPU)

## Overview
Predicts **5-day-ahead IV Rank direction** for options entry timing.

- **Features**: 30 days of rolling windows: `HV_5`, `HV_20`, `HV_60`, `RSI_14`, `momentum_5d`, `momentum_20d`, `price_vol_ratio`, `hv_ratio_5_20`
- **Model**: LSTM (2 layers, 64 hidden units) → binary classification
- **Target**: `will_IV_increase` — 1 if IV rank will be higher in 5 days, 0 if lower/flat
- **Use case**: Only enter iron condors / selling strategies when model predicts IV direction = **decreasing** (confirming IV is near peak)

## Kaggle Instructions
1. Upload this notebook to [kaggle.com/code](https://kaggle.com/code)
2. Add your Alpaca credentials as Kaggle Secrets: `ALPACA_KEY` and `ALPACA_SECRET`
3. Enable T4 GPU accelerator (Settings → Accelerator → GPU T4 x2)
4. Run All → download `iv_predictor.pt` from the output files panel
5. Copy `iv_predictor.pt` to `backend/app/ml/models/iv_predictor.pt`
6. Set `IV_PREDICTOR_PATH=app/ml/models/iv_predictor.pt` in your `.env`

## Expected Performance
- Validation accuracy: ~62-68% (binary classification of IV direction)
- Using TimeSeriesSplit to prevent data leakage
- Walk-forward validation only — no in-sample testing


In [ ]:
# Install dependencies (uncomment for Kaggle/Colab)
# !pip install torch pandas numpy scikit-learn matplotlib httpx --quiet

import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, classification_report
import httpx

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

In [ ]:
# ── Alpaca credentials ─────────────────────────────────────────────────────
# On Kaggle: use Kaggle Secrets (Add-ons > Secrets)
# On Colab:  use userdata.get() or os.environ

try:
    # Kaggle Secrets
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    ALPACA_KEY = secrets.get_secret("ALPACA_KEY")
    ALPACA_SECRET = secrets.get_secret("ALPACA_SECRET")
except ImportError:
    # Local or Colab: set env vars or fill in directly (never commit credentials)
    ALPACA_KEY = os.environ.get("ALPACA_KEY", "")
    ALPACA_SECRET = os.environ.get("ALPACA_SECRET", "")

ALPACA_DATA_BASE = "https://data.alpaca.markets"

# Symbols to train on
SYMBOLS = ["SPY", "QQQ", "AAPL", "MSFT", "NVDA", "TSLA", "AMZN", "META"]

# Sequence length for LSTM (30 trading days = ~6 weeks of daily bars)
SEQ_LEN = 30

# Forecast horizon: predict IV rank direction 5 days ahead
HORIZON = 5

print(f"Alpaca key configured: {'YES' if ALPACA_KEY else 'NO — fill in credentials!'}")

In [ ]:
def fetch_bars(symbol: str, start: str = "2020-01-01") -> pd.DataFrame:
    """Fetch daily bars from Alpaca for a symbol.
    
    Returns DataFrame with columns: o, h, l, c, v
    Uses IEX feed (free, no subscription required).
    """
    url = f"{ALPACA_DATA_BASE}/v2/stocks/{symbol}/bars"
    headers = {
        "APCA-API-KEY-ID": ALPACA_KEY,
        "APCA-API-SECRET-KEY": ALPACA_SECRET,
    }
    params = {
        "timeframe": "1Day",
        "start": start,
        "limit": 1500,
        "feed": "iex",
        "adjustment": "all",
    }
    
    all_bars = []
    next_token = None
    
    while True:
        if next_token:
            params["page_token"] = next_token
        resp = httpx.get(url, params=params, headers=headers, timeout=30)
        data = resp.json()
        bars = data.get("bars", [])
        all_bars.extend(bars)
        next_token = data.get("next_page_token")
        if not next_token or not bars:
            break
    
    if not all_bars:
        raise ValueError(f"No bars returned for {symbol}")
    
    df = pd.DataFrame(all_bars)
    df["t"] = pd.to_datetime(df["t"])
    df = df.set_index("t").sort_index()
    # Rename columns to lowercase
    df.columns = [c.lower() for c in df.columns]
    return df[["o", "h", "l", "c", "v"]].rename(columns={"o": "open", "h": "high", "l": "low", "c": "close", "v": "volume"})


# Download all symbols
print("Fetching historical data from Alpaca...")
all_data: dict[str, pd.DataFrame] = {}
for sym in SYMBOLS:
    try:
        all_data[sym] = fetch_bars(sym)
        print(f"  {sym}: {len(all_data[sym])} bars ({all_data[sym].index[0].date()} → {all_data[sym].index[-1].date()})")
    except Exception as e:
        print(f"  {sym}: FAILED — {e}")

print(f"\nLoaded {len(all_data)} symbols successfully.")

In [ ]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create features for IV rank direction prediction.
    
    Features (8 total):
    - hv5:           5-day historical volatility (annualised)
    - hv20:          20-day historical volatility (annualised)
    - hv60:          60-day historical volatility (annualised)
    - hv_ratio_5_20: hv5/hv20 ratio (rising = vol expanding short-term)
    - rsi:           RSI(14)
    - mom5:          5-day price momentum
    - mom20:         20-day price momentum
    - price_vol:     price × volume normalised (liquidity proxy)
    
    Target:
    - target: 1 if IV rank will be higher in HORIZON days, else 0
    """
    feat = df.copy()
    log_ret = np.log(feat["close"] / feat["close"].shift(1))
    
    # Historical volatility at different lookbacks (annualised)
    feat["hv5"]  = log_ret.rolling(5).std()  * np.sqrt(252)
    feat["hv20"] = log_ret.rolling(20).std() * np.sqrt(252)
    feat["hv60"] = log_ret.rolling(60).std() * np.sqrt(252)
    
    # Short vs medium vol ratio (rising = volatility expanding near-term)
    feat["hv_ratio_5_20"] = feat["hv5"] / (feat["hv20"] + 1e-8)
    
    # IV Rank proxy (52-week HV20 percentile), 0-100
    hv_min = feat["hv20"].rolling(252).min()
    hv_max = feat["hv20"].rolling(252).max()
    feat["iv_rank"] = (feat["hv20"] - hv_min) / (hv_max - hv_min + 1e-8) * 100
    
    # Momentum (simple price change)
    feat["mom5"]  = feat["close"].pct_change(5)
    feat["mom20"] = feat["close"].pct_change(20)
    
    # RSI(14)
    delta = feat["close"].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    feat["rsi"] = 100 - (100 / (1 + gain / (loss + 1e-8)))
    
    # Price-volume (liquidity proxy, normalised)
    pv = feat["close"] * feat["volume"]
    feat["price_vol"] = pv / pv.rolling(20).mean().clip(lower=1)
    
    # Target: will IV rank be higher in HORIZON days? 1=yes, 0=no
    feat["target"] = (feat["iv_rank"].shift(-HORIZON) > feat["iv_rank"]).astype(int)
    
    return feat.dropna()


# Build features for all symbols
FEATURE_COLS = ["hv5", "hv20", "hv60", "hv_ratio_5_20", "rsi", "mom5", "mom20", "price_vol"]

print("Engineering features...")
all_features: dict[str, pd.DataFrame] = {}
for sym, df in all_data.items():
    feat = make_features(df)
    all_features[sym] = feat
    print(f"  {sym}: {len(feat)} samples, target balance = {feat['target'].mean():.2%} positive")

# Concatenate all symbols for training
combined = pd.concat(all_features.values(), ignore_index=True)
print(f"\nTotal samples: {len(combined):,}")
print(f"Overall target balance: {combined['target'].mean():.2%} positive")

In [ ]:
def make_sequences(features: np.ndarray, targets: np.ndarray, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    """Convert flat feature array into overlapping LSTM sequences.
    
    Returns:
        X: shape (n_samples, seq_len, n_features)
        y: shape (n_samples,)
    """
    X, y = [], []
    for i in range(seq_len, len(features)):
        X.append(features[i - seq_len: i])
        y.append(targets[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


class IVPredictor(nn.Module):
    """2-layer LSTM for IV rank direction prediction.
    
    Architecture:
    - Input: (batch, seq_len=30, features=8)
    - LSTM: 2 layers, 64 hidden units, dropout=0.2
    - FC: 64 → 1
    - Sigmoid: binary output (probability IV rank increases)
    """
    
    def __init__(self, input_size: int = 8, hidden_size: int = 64, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, input_size)
        lstm_out, (hn, _) = self.lstm(x)
        # hn[-1]: last layer hidden state (batch, hidden_size)
        out = self.dropout(hn[-1])
        return self.sigmoid(self.fc(out)).squeeze(-1)  # (batch,)


print("Model architecture:")
model_test = IVPredictor(input_size=len(FEATURE_COLS))
print(model_test)
total_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

In [ ]:
def train_model(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    epochs: int = 50,
    batch_size: int = 64,
    lr: float = 0.001,
    patience: int = 8,
) -> tuple[IVPredictor, dict]:
    """Train IVPredictor with early stopping and AdamW optimizer.
    
    Returns trained model and training history.
    """
    model = IVPredictor(input_size=X_train.shape[-1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=4, factor=0.5)
    criterion = nn.BCELoss()
    
    X_tr = torch.FloatTensor(X_train).to(DEVICE)
    y_tr = torch.FloatTensor(y_train).to(DEVICE)
    X_v  = torch.FloatTensor(X_val).to(DEVICE)
    y_v  = torch.FloatTensor(y_val).to(DEVICE)
    
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_state = None
    no_improve = 0
    
    n_train = len(X_tr)
    
    for epoch in range(epochs):
        # ── Training ────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        indices = torch.randperm(n_train)
        
        for start in range(0, n_train, batch_size):
            batch_idx = indices[start: start + batch_size]
            xb, yb = X_tr[batch_idx], y_tr[batch_idx]
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * len(xb)
        
        train_loss /= n_train
        
        # ── Validation ──────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_preds = model(X_v)
            val_loss = criterion(val_preds, y_v).item()
            val_acc = ((val_preds > 0.5).float() == y_v).float().mean().item()
        
        scheduler.step(val_loss)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        
        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch+1:3d}/{epochs}  "
                  f"train_loss={train_loss:.4f}  "
                  f"val_loss={val_loss:.4f}  "
                  f"val_acc={val_acc:.4f}  "
                  f"best={best_val_acc:.4f}")
        
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            break
    
    # Restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
    
    print(f"\nBest validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.1f}%)")
    return model, history

In [ ]:
# ── Walk-Forward Training with TimeSeriesSplit ─────────────────────────────
# Use SPY as the primary training symbol (most liquid, longest history)
# Then validate on out-of-sample periods from other symbols

# Prepare dataset from all symbols combined
raw_features = combined[FEATURE_COLS].values.astype(np.float32)
raw_targets  = combined["target"].values.astype(np.float32)

# Standardize features (fit on training data only to prevent leakage)
scaler = StandardScaler()

# Use final 80% for training, last 20% as holdout test
split_idx = int(len(raw_features) * 0.80)

train_feat = raw_features[:split_idx]
test_feat  = raw_features[split_idx:]
train_tgt  = raw_targets[:split_idx]
test_tgt   = raw_targets[split_idx:]

# Fit scaler on training data only
train_feat_scaled = scaler.fit_transform(train_feat)
test_feat_scaled  = scaler.transform(test_feat)

# Build sequences
X_train_seq, y_train_seq = make_sequences(train_feat_scaled, train_tgt, SEQ_LEN)
X_test_seq,  y_test_seq  = make_sequences(test_feat_scaled,  test_tgt,  SEQ_LEN)

print(f"Train sequences: {X_train_seq.shape}")
print(f"Test sequences:  {X_test_seq.shape}")
print(f"Train target balance: {y_train_seq.mean():.2%}")
print(f"Test target balance:  {y_test_seq.mean():.2%}")

# Further split train into train/val (last 15% of train as val)
val_split = int(len(X_train_seq) * 0.85)
X_tr = X_train_seq[:val_split]
X_val = X_train_seq[val_split:]
y_tr = y_train_seq[:val_split]
y_val = y_train_seq[val_split:]

print(f"\nFinal split:")
print(f"  Train: {X_tr.shape[0]:,} sequences")
print(f"  Val:   {X_val.shape[0]:,} sequences")
print(f"  Test:  {X_test_seq.shape[0]:,} sequences (held out)")

In [ ]:
# ── Train the model ────────────────────────────────────────────────────────
print("Starting training...")
print("=" * 60)

model, history = train_model(
    X_tr, y_tr,
    X_val, y_val,
    epochs=60,
    batch_size=128,
    lr=0.001,
    patience=10,
)

# ── Plot training curves ────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], label="Train Loss", color="#f5a623")
ax1.plot(history["val_loss"],   label="Val Loss",   color="#00c853")
ax1.set_title("Loss Curves")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("BCE Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history["val_acc"], label="Val Accuracy", color="#2196f3")
ax2.axhline(y=0.5, color="red", linestyle="--", alpha=0.5, label="Random baseline")
ax2.set_title("Validation Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png")

In [ ]:
# ── Evaluate on held-out test set ──────────────────────────────────────────
model.eval()
with torch.no_grad():
    X_test_t = torch.FloatTensor(X_test_seq).to(DEVICE)
    test_probs = model(X_test_t).cpu().numpy()
    test_preds = (test_probs > 0.5).astype(int)

test_acc = accuracy_score(y_test_seq.astype(int), test_preds)
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")
print()
print("Classification Report:")
print(classification_report(
    y_test_seq.astype(int), test_preds,
    target_names=["IV_Decreasing", "IV_Increasing"]
))

# Calibration check: are probabilities well-calibrated?
bins = np.linspace(0, 1, 11)
bin_accs = []
for low, high in zip(bins[:-1], bins[1:]):
    mask = (test_probs >= low) & (test_probs < high)
    if mask.sum() > 0:
        bin_accs.append((low + high) / 2, y_test_seq[mask].mean())

print("\nModel is ready for deployment when test accuracy > 58%")

In [ ]:
# ── Save model + scaler ────────────────────────────────────────────────────
import pickle

MODEL_PATH  = "iv_predictor.pt"
SCALER_PATH = "iv_predictor_scaler.pkl"

# Save model weights + metadata
torch.save({
    "model_state_dict": model.state_dict(),
    "model_config": {
        "input_size":  len(FEATURE_COLS),
        "hidden_size": 64,
        "num_layers":  2,
        "dropout":     0.2,
    },
    "feature_cols": FEATURE_COLS,
    "seq_len":      SEQ_LEN,
    "horizon_days": HORIZON,
    "test_accuracy": float(test_acc),
    "symbols_trained_on": SYMBOLS,
}, MODEL_PATH)

# Save scaler (needed for inference)
with open(SCALER_PATH, "wb") as f:
    pickle.dump(scaler, f)

print(f"Model saved to: {MODEL_PATH}")
print(f"Scaler saved to: {SCALER_PATH}")
print(f"Test accuracy: {test_acc*100:.1f}%")
print()
print("Next steps:")
print("  1. Download iv_predictor.pt and iv_predictor_scaler.pkl from Kaggle output")
print("  2. Copy to backend/app/ml/models/")
print("  3. Set IV_PREDICTOR_PATH=app/ml/models/iv_predictor.pt in .env")
print("  4. The ml_enhanced iron condor strategy will use this model as a filter")

In [ ]:
# ── Example: Load and use model for inference ──────────────────────────────
# This shows exactly how the backend will use the model in production.

import pickle

def load_iv_predictor(model_path: str, scaler_path: str) -> tuple:
    """Load trained IV predictor from disk.
    
    Returns (model, scaler, config)
    """
    checkpoint = torch.load(model_path, map_location="cpu")
    config = checkpoint["model_config"]
    loaded_model = IVPredictor(**config)
    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    loaded_model.eval()
    
    with open(scaler_path, "rb") as f:
        loaded_scaler = pickle.load(f)
    
    return loaded_model, loaded_scaler, checkpoint


def predict_iv_direction(model, scaler, recent_df: pd.DataFrame, seq_len: int = 30) -> dict:
    """Predict whether IV rank will increase in the next 5 days.
    
    Args:
        model:     loaded IVPredictor
        scaler:    fitted StandardScaler
        recent_df: DataFrame with at least seq_len + buffer rows, columns: open, high, low, close, volume
        seq_len:   sequence length the model was trained with (default 30)
    
    Returns dict with:
        - prob_increase: float 0-1 (probability IV rank increases)
        - direction: 'increasing' | 'decreasing'
        - sell_premium_ok: bool (True = IV likely near peak, good time to sell)
    """
    feat = make_features(recent_df)
    if len(feat) < seq_len:
        return {"error": f"Need at least {seq_len} rows after feature engineering"}
    
    feature_cols = ["hv5", "hv20", "hv60", "hv_ratio_5_20", "rsi", "mom5", "mom20", "price_vol"]
    last_seq = feat[feature_cols].values[-seq_len:].astype(np.float32)
    last_seq_scaled = scaler.transform(last_seq)
    
    x = torch.FloatTensor(last_seq_scaled).unsqueeze(0)  # (1, seq_len, features)
    with torch.no_grad():
        prob = model(x).item()
    
    return {
        "prob_increase": round(prob, 4),
        "prob_decrease": round(1 - prob, 4),
        "direction": "increasing" if prob > 0.5 else "decreasing",
        # Sell premium when model says IV is near peak (about to decrease)
        "sell_premium_ok": prob < 0.45,
        "confidence": round(abs(prob - 0.5) * 2, 4),  # 0=uncertain, 1=very confident
    }


# ── Quick test with saved model ────────────────────────────────────────────
loaded_model, loaded_scaler, ckpt = load_iv_predictor(MODEL_PATH, SCALER_PATH)
print(f"Loaded model (test_acc={ckpt['test_accuracy']*100:.1f}%)")

# Test on SPY
if "SPY" in all_data:
    spy_df = all_data["SPY"]
    result = predict_iv_direction(loaded_model, loaded_scaler, spy_df)
    print(f"\nSPY IV Prediction:")
    for k, v in result.items():
        print(f"  {k}: {v}")
    print()
    print(f"  → {'SELL PREMIUM OK' if result.get('sell_premium_ok') else 'AVOID — IV may rise further'}")

print("\nDone! Upload iv_predictor.pt and iv_predictor_scaler.pkl to backend/app/ml/models/")